# WSC 2026 — REVISION notebook: German Credit

**This is the revised pipeline for the major-revision response.** It reuses the
original seeds (42), stratified 80/20 split, and architecture ([256,128,64], ReLU,
dropout 0.2) so results stay comparable to Table 1. New: scenario-based simulation
study (P1), teacher tuning + AUC-anomaly ablation (P2a), EO/EOdds metrics + EO penalty
(P2b), batched throughput (P2c). All logic lives in `revision_outputs/rtxfair_sim/`.


In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
import sys; sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np
from rtxfair_sim import sim, figures, metrics as M
from rtxfair_sim.data import load_dataset
from rtxfair_sim.model import train_teacher, teacher_shap, FairStudentXAI
from rtxfair_sim.train import train_student
from rtxfair_sim.config import DATASETS, ScenarioConfig, PENALTY_DP, PENALTY_EO
NAME = "german"
cfg = DATASETS[NAME]; print(cfg)

## 0. Data (reads csvs/ raw files; reconstructs the ucimlrepo schema)

In [ ]:
b = load_dataset(NAME)
print('train', b.X_train.shape, 'protected g1=', int(b.sensitive_train.sum()),
      'g0=', int((b.sensitive_train==0).sum()))

## 1. Baseline reproduction (teacher + baseline student), with EO/EOdds added

In [ ]:
teacher = train_teacher(b.X_train, b.y_train, 100, 4, 0.1)
from sklearn.metrics import roc_auc_score
print('fixed teacher AUC', roc_auc_score(b.y_test, teacher.predict_proba(b.X_test)[:,1]))
sv_tr, sv_te, _ = teacher_shap(teacher, b.X_train, b.X_test)
sc = ScenarioConfig(dataset=NAME, seed=42, lambda_fair=cfg.lambda_fair,
                    epochs=cfg.epochs, batch_size=cfg.batch_size, lr=cfg.lr)
rec, model = train_student(b.X_train, b.y_train, sv_tr, b.sensitive_train,
                           b.X_test, b.y_test, sv_te, b.sensitive_test, sc)
print({k: round(rec[k],4) for k in ['AUC','SHAP_R2','DP_gap','EO_gap','EOdds_gap','p99_latency_ms']})

## P1. Scenario-based simulation study (OFAT + factorial, 5 seeds)
Run from the shell for long jobs: `python ../run_chunk.py german` (repeat until REMAINING 0).
On a normal multi-core/GPU machine you can run inline:

In [ ]:
# Inline run (German finishes in ~10 min; for creditcard/bank prefer run_chunk.py).
scenarios = sim.build_ofat_scenarios(NAME) + sim.build_factorial_scenarios(NAME)
# df = sim.run_scenarios(NAME, scenarios, out_csv=f'../results/{NAME}_runs.csv')
df = pd.read_csv(f'../results/{NAME}_runs.csv')  # load precomputed if present
print(len(df), 'runs')

In [ ]:
def sub(p): return df[df.label.str.startswith(p)]
agg_lambda = sim.aggregate(sub('OFAT:lambda_fair'), ['lambda_fair'])
agg_minor  = sim.aggregate(sub('OFAT:minority_fraction'), ['minority_fraction'])
agg_batch  = sim.aggregate(sub('OFAT:batch_size'), ['batch_size'])
display(agg_lambda[['lambda_fair','n_rep','AUC_mean','AUC_ci95','DP_gap_mean','EO_gap_mean']].round(4))

In [ ]:
figures.dual_response_vs_lambda(agg_lambda, f'../figures/{NAME}_lambda_response', cfg.pretty)
figures.pareto_auc_dpgap(agg_lambda, f'../figures/{NAME}_pareto', cfg.pretty)
figures.stability_vs_minority(agg_minor, f'../figures/{NAME}_stability_minority', cfg.pretty)
print('figures written to ../figures/')

## P2a. Teacher tuning + AUC-anomaly ablation
`python ../run_p2a.py german`  (RandomizedSearch via manual CV; multi-seed λ=0 vs λ=baseline).

## P2b. EO / Equalized-Odds penalty variant
Same setup, `penalty=PENALTY_EO` (TPR-difference surrogate, same safety net).

In [ ]:
sc_eo = ScenarioConfig(dataset=NAME, seed=42, lambda_fair=cfg.lambda_fair,
                       epochs=cfg.epochs, batch_size=cfg.batch_size, lr=cfg.lr,
                       penalty=PENALTY_EO)
rec_eo, _ = train_student(b.X_train, b.y_train, sv_tr, b.sensitive_train,
                          b.X_test, b.y_test, sv_te, b.sensitive_test, sc_eo)
print('DP-penalty :', {k: round(rec[k],4)    for k in ['AUC','DP_gap','EO_gap','EOdds_gap']})
print('EO-penalty :', {k: round(rec_eo[k],4) for k in ['AUC','DP_gap','EO_gap','EOdds_gap']})

## P2c. Batched-inference throughput (device noted)

In [ ]:
rows = M.batched_throughput(model, b.X_test, batch_sizes=(1,32,128,512,4096), device='cpu')
display(pd.DataFrame(rows))